# 2 — Extração de Características

Executa `extract_features` sobre todos os grafos e calcula χ(G), construindo o `dataset.csv`.

**Pré-requisito:** executar `1 - carregamento_e_rotulacao.ipynb` (variáveis `all_graphs`, `chromatic_number_backtracking` devem estar em memória, ou re-execute as células abaixo).

**Saída:** `data/processed/dataset.csv`

In [1]:
import threading
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TIMEOUT_SEC  = 60

np.random.seed(RANDOM_STATE)

ROOT_DIR      = Path('.')
PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH  = PROCESSED_DIR / 'dataset.csv'

FEATURE_COLS = [
    'n_vertices', 'n_edges', 'density',
    'min_degree', 'max_degree', 'mean_degree', 'std_degree',
    'avg_clustering', 'diameter', 'is_disconnected',
    'avg_betweenness', 'largest_clique', 'n_components',
]

print('Setup concluído.')

Setup concluído.


## Re-importação (caso execute este notebook isoladamente)

Se você já executou o notebook 1 nesta sessão do Jupyter, as funções já estão em memória. Caso contrário, execute as células a seguir.

In [2]:
if 'all_graphs' not in dir() or 'chromatic_number_backtracking' not in dir():
    print('Executando notebook 1...')
    %run "1 - carregamento_e_rotulacao.ipynb"
    print('Notebook 1 concluído.')

Executando notebook 1...
Setup concluído.
  Instâncias DIMACS: C:\Users\Pedro\Desktop\PPGCC21_RP\data\dimacs
Grafos sintéticos gerados: 484
Nenhum .col encontrado em data\dimacs.
Continuando apenas com grafos sintéticos.
Total de instâncias: 484
Notebook 1 concluído.


## Função de Extração de Características

In [3]:
def _find_largest_clique(G_s, timeout=10):
    """Executa find_cliques com timeout. Retorna tamanho ou None se exceder."""
    result    = [None]
    stop_flag = threading.Event()

    def run():
        best = 1
        try:
            for clique in nx.find_cliques(G_s):
                if stop_flag.is_set():
                    return
                if len(clique) > best:
                    best = len(clique)
            result[0] = best
        except Exception:
            result[0] = best

    t = threading.Thread(target=run, daemon=True)
    t.start()
    t.join(timeout)
    if t.is_alive():
        stop_flag.set()
        return result[0] if result[0] is not None else 1
    return result[0]


def extract_features(G, clique_timeout=10):
    """Extrai vetor de 13 características estruturais e topológicas."""
    G_s = nx.Graph(G)
    G_s.remove_edges_from(nx.selfloop_edges(G_s))
    G_s = nx.convert_node_labels_to_integers(G_s)

    n = G_s.number_of_nodes()
    if n == 0:
        return None

    m       = G_s.number_of_edges()
    degrees = [d for _, d in G_s.degree()]
    is_conn = nx.is_connected(G_s)
    n_comp  = nx.number_connected_components(G_s)

    if is_conn:
        if n <= 500:
            diameter = nx.diameter(G_s)
        else:
            sources  = list(np.random.choice(n, min(10, n), replace=False))
            diameter = max(
                max(nx.single_source_shortest_path_length(G_s, s).values())
                for s in sources
            )
    else:
        diameter = -1

    k_bc = min(50, n) if n > 200 else None
    bc   = nx.betweenness_centrality(G_s, k=k_bc, normalized=True, seed=RANDOM_STATE)

    largest_clique = _find_largest_clique(G_s, timeout=clique_timeout)

    return {
        'n_vertices'     : n,
        'n_edges'        : m,
        'density'        : nx.density(G_s),
        'min_degree'     : int(np.min(degrees)),
        'max_degree'     : int(np.max(degrees)),
        'mean_degree'    : float(np.mean(degrees)),
        'std_degree'     : float(np.std(degrees)),
        'avg_clustering' : nx.average_clustering(G_s),
        'diameter'       : diameter,
        'is_disconnected': int(not is_conn),
        'avg_betweenness': float(np.mean(list(bc.values()))),
        'largest_clique' : largest_clique,
        'n_components'   : n_comp,
    }

## Construção do Dataset

In [4]:
def build_dataset(graphs, timeout=TIMEOUT_SEC):
    rows      = []
    skipped   = 0
    timed_out = 0

    bar = tqdm(graphs, desc='Aguardando...', unit='grafo', dynamic_ncols=True)

    for gtype, G in bar:
        n_v = G.number_of_nodes()
        n_e = G.number_of_edges()

        bar.set_description(f'[{gtype}] n={n_v} m={n_e} | features')
        feats = extract_features(G)
        if feats is None:
            skipped += 1
            bar.set_postfix(aceitos=len(rows), skip=skipped, timeout=timed_out)
            continue

        bar.set_description(f'[{gtype}] n={n_v} m={n_e} | χ(G)')
        chi = chromatic_number_backtracking(G, timeout=timeout)
        if chi is None:
            timed_out += 1
            bar.set_postfix(aceitos=len(rows), skip=skipped, timeout=timed_out)
            continue

        row = {'graph_type': gtype, 'chi': chi}
        row.update(feats)
        rows.append(row)
        bar.set_postfix(aceitos=len(rows), skip=skipped, timeout=timed_out)

    bar.set_description('Concluído')
    df = pd.DataFrame(rows)
    print(f'\nDataset: {len(df)} instâncias | skip={skipped} | timeout={timed_out}')
    return df


if not DATASET_PATH.exists():
    print('Construindo dataset (pode demorar)...')
    t0 = time.time()
    df = build_dataset(all_graphs)
    df.to_csv(DATASET_PATH, index=False)
    print(f'Salvo em {DATASET_PATH}  ({time.time()-t0:.1f}s)')
else:
    print(f'Carregando cache: {DATASET_PATH}')
    df = pd.read_csv(DATASET_PATH)
    print(f'  {len(df)} instâncias.')

Construindo dataset (pode demorar)...


Aguardando...:   0%|          | 0/484 [00:00<?, ?grafo/s]


Dataset: 440 instâncias | skip=0 | timeout=44
Salvo em data\processed\dataset.csv  (2770.3s)


In [5]:
print(df[['chi'] + FEATURE_COLS].describe().round(3).to_string())
df.head(10)

           chi  n_vertices   n_edges  density  min_degree  max_degree  mean_degree  std_degree  avg_clustering  diameter  is_disconnected  avg_betweenness  largest_clique  n_components
count  440.000     440.000   440.000  440.000      440.00     440.000      440.000     440.000         440.000   440.000          440.000          440.000         440.000       440.000
mean     3.832      21.564    70.189    0.346        3.72       8.241        5.374       1.196           0.221     4.159            0.084            0.088           3.657         1.168
std      3.141      14.637   111.377    0.268        4.94       7.025        5.590       0.985           0.299     3.945            0.278            0.085           3.119         0.684
min      1.000       2.000     0.000    0.000        0.00       0.000        0.000       0.000           0.000    -1.000            0.000            0.000           1.000         1.000
25%      2.000      10.000    15.000    0.128        1.00       4.000      

,graph_type,chi,n_vertices,n_edges,density,min_degree,max_degree,mean_degree,std_degree,avg_clustering,diameter,is_disconnected,avg_betweenness,largest_clique,n_components
0,random_er,2,5,1,0.1,0,1,0.4,0.489898,0.000000,-1,1,0.000000,2,4
1,random_er,1,5,0,0.0,0,0,0.0,0.000000,0.000000,-1,1,0.000000,1,5
2,random_er,2,5,1,0.1,0,1,0.4,0.489898,0.000000,-1,1,0.000000,2,4
3,random_er,2,5,2,0.2,0,1,0.8,0.400000,0.000000,-1,1,0.000000,2,3
4,random_er,2,5,2,0.2,0,1,0.8,0.400000,0.000000,-1,1,0.000000,2,3
5,random_er,2,5,2,0.2,0,1,0.8,0.400000,0.000000,-1,1,0.000000,2,3
6,random_er,2,5,1,0.1,0,1,0.4,0.489898,0.000000,-1,1,0.000000,2,4
7,random_er,3,5,4,0.4,0,3,1.6,1.019804,0.466667,-1,1,0.066667,3,2
8,random_er,2,5,1,0.1,0,1,0.4,0.489898,0.000000,-1,1,0.000000,2,4
9,random_er,3,5,6,0.6,2,4,2.4,0.800000,0.866667,2,0,0.133333,3,1
